### 1. Installation des dépendances Python
Une seule cellule d'installation, incluant `mcp-server-git` (il manquait dans le fichier d'origine — c'est ce paquet qui fournit le module `mcp_server_git` utilisé plus bas pour le serveur MCP "git").

In [ ]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "nest_asyncio" \
  "mcp-server-git"


### 2. Installation de Node.js / npx
Nécessaire pour lancer le serveur MCP `@modelcontextprotocol/server-filesystem` via `npx`.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y nodejs npm
!node --version
!npx --version


### 3. Connect to MCP servers from your agent runtime
Use `MultiServerMCPClient` to register servers. For this, we'll first define a `WORKDIR`.

**Erreur corrigée :** dans Colab/Jupyter, `sys.stderr` est remplacé par un objet `ipykernel.iostream.OutStream` qui n'implémente pas `fileno()`. Or `stdio_client` (utilisé en interne par `langchain-mcp-adapters`) transmet `sys.stderr` directement à `subprocess.Popen` comme flux d'erreur du sous-processus, ce qui exige un vrai descripteur de fichier — d'où le `UnsupportedOperation: fileno`. La correction consiste à basculer temporairement `sys.stderr` vers `sys.__stderr__` (le flux natif, qui a un `fileno()` valide) le temps de créer la connexion.

In [ ]:
import asyncio
import os
import subprocess
import sys

import nest_asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

nest_asyncio.apply()

WORKDIR = "/content"  # Define the working directory for the servers
os.makedirs(WORKDIR, exist_ok=True)

# Le serveur MCP "git" a besoin d'un dépôt git existant dans WORKDIR.
if not os.path.isdir(os.path.join(WORKDIR, ".git")):
    subprocess.run(["git", "init", WORKDIR], check=True)

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    "git": {
        "transport": "stdio",
        "command": sys.executable,  # garantit le bon interpréteur Python (plutôt que "python", absent sur certains environnements)
        "args": ["-m", "mcp_server_git", "--repository", WORKDIR],
    },
}

# Redirection temporaire de sys.stderr vers le flux natif pour éviter l'erreur fileno()
original_stderr = sys.stderr
try:
    sys.stderr = sys.__stderr__
    client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)
    tools = asyncio.get_event_loop().run_until_complete(client.get_tools())
finally:
    sys.stderr = original_stderr  # restauration du flux d'origine

print("Tool count:", len(tools))
print([t.name for t in tools])
